# 02. Mocking

**Tier:** Intermediate. **Why you'd use this:** A function reads an environment variable or today's date. Tests should not depend on your laptop's clock or shell. `monkeypatch` replaces those seams for one test.

**After this lesson you can:** use `monkeypatch.setenv` and `monkeypatch.setattr`; stub a function; keep IO out of unit tests.
About 25 minutes. **Uses:** [pytest](00-pytest-lesson.ipynb), [Tests](01-unittest-lesson.ipynb)


## Quick review

1. What belongs in arrange-act-assert?
2. Why avoid shared writable globals in tests?

<details><summary>Answers</summary>

Setup, call, check. They create order-dependent failures.
</details>


## The idea

A mock replaces a real dependency with a controlled stand-in. pytest's `monkeypatch` fixture patches env vars, attributes, and dict entries for the duration of one test, then undoes them. That keeps tests about your logic, not about the network or clock.

<details><summary>Explain more</summary>

`monkeypatch.setenv("MODE", "test")` makes `os.environ["MODE"]` read `"test"` inside that test only.
</details>


## See it work

Patch an environment variable while calling a helper.


In [1]:
import os


def mode_label() -> str:
    # 1. Production reads the real environment.
    return os.environ.get("APP_MODE", "live")


class Patch:
    def __init__(self):
        self._old = os.environ.get("APP_MODE")

    def setenv(self, key, value):
        os.environ[key] = value

    def undo(self):
        if self._old is None:
            os.environ.pop("APP_MODE", None)
        else:
            os.environ["APP_MODE"] = self._old


patch = Patch()
patch.setenv("APP_MODE", "test")
print(mode_label())
patch.undo()
print(mode_label())

test
live


## Check yourself

Answer these before you run the cell.

1. What should a unit test replace when the code calls `datetime.now`?
2. Does `monkeypatch` persist after the test ends?


In [2]:
os.environ["APP_MODE"] = "demo"
print(os.environ["APP_MODE"])
os.environ.pop("APP_MODE", None)
print("APP_MODE" in os.environ)

demo
False


<details><summary>Answers</summary>

The clock (or inject a clock argument). No. It restores the old values.
</details>


## Why it works this way

Unit tests should fail for your logic, not for flaky surroundings. Seams such as `os.environ`, `Path.read_text`, or a client attribute are replaceable. `monkeypatch` performs the replacement and always rolls it back, which prevents leaks into later tests. That rollback is the difference between a helper stub and a landmine.

Prefer dependency injection when you design new code: pass a clock or reader as an argument. Use monkeypatch when the seam is already a module attribute. Attribute patches should target the name the system under test looks up. Patching where it is defined, not where it is used, is a common miss. Read the import style before you patch.

Mocking too much produces tests that mirror implementation and break on every refactor. Mock at the boundary, assert on your code's decisions. Cost of a patch is small. Cost of over-mocking is brittle suites.

Evidence: setting and clearing an env var around `mode_label`, and a subprocess pytest example that uses the real `monkeypatch` fixture in a tiny file. Never leave a patch in place at import time. Never mock the system under test itself so thoroughly that the test cannot fail.

`monkeypatch.setattr` can replace callables, constants, and class methods for one test. Prefer patching the lookup site the code uses after import. If module A does `from b import fn`, patching `b.fn` will not change A's binding; patch `A.fn` instead.

Autospec and full mock objects are available in `unittest.mock`, and pytest integrates with them. Start with monkeypatch for simple seams. Reach for `MagicMock` when you need call history. Still assert on outcomes your users care about, not on every internal call, unless the interaction is the requirement.


In [3]:
import subprocess
import sys
from pathlib import Path

lines = [
    "import os",
    "",
    "def mode_label():",
    '    return os.environ.get("APP_MODE", "live")',
    "",
    "def test_mode(monkeypatch):",
    '    monkeypatch.setenv("APP_MODE", "test")',
    '    assert mode_label() == "test"',
    "",
]
sample = Path("exercise/_patch_demo.py")
sample.write_text("\n".join(lines), encoding="utf-8")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(sample)],
    capture_output=True,
    text=True,
)
print(result.stdout.strip())
print(result.returncode)
sample.unlink()

.                                                                        [100%]
1 passed in 0.01s
0


## When to use it

Use monkeypatch for env, time, and thin IO seams in unit tests. Use real tmp files for filesystem behaviour you own. Don't mock every collaborator. Don't test the mock's configuration instead of your code. Leave a comment at the patch when the seam is non-obvious so future readers know why it exists.


## Common mistake

Patching a value in one test and forgetting that the next test still sees it when you patched by hand without cleanup.


In [4]:
import os

os.environ["APP_MODE"] = "test"
print(os.environ["APP_MODE"])
print("leak risk without restore")
os.environ.pop("APP_MODE", None)

test
leak risk without restore


'test'

Use `monkeypatch.setenv` in pytest so restore is automatic.


In [5]:
print("def test_mode(monkeypatch):")
print('    monkeypatch.setenv("APP_MODE", "test")')
print("    assert mode_label() == 'test'")

def test_mode(monkeypatch):
    monkeypatch.setenv("APP_MODE", "test")
    assert mode_label() == 'test'


## Try it

Patch `Path.read_text` on a module under test to return fixed CSV text. Which assert proves your parser, not the filesystem, ran?


## Recap

- Mock at boundaries. Prefer injection when you can.
- `monkeypatch` restores after each test.
- Over-mocking tests the mock, not your code.

**Next:** [practice](02-mocking-practice.ipynb)
